# 4. Model Pruning

This notebook performs pruning on the LoRA-finetuned model.

**Workflow:**
- Load the LoRA-finetuned model checkpoint from `../checkpoints/model_lora`.
- Apply L1 unstructured pruning to all `torch.nn.Linear` layers (pruning 30% of weights).
- Save the pruned model checkpoint in `../checkpoints/model_pruned`.

**Constraints:**
- Model loaded in float16 for memory efficiency.
- Device selection (MPS, CUDA, or CPU) is respected.


In [1]:
import os
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# Helper function to choose the appropriate device (MPS > CUDA > CPU)
def get_device():
    if torch.backends.mps.is_available():
        device = torch.device("mps")
        print("Device Selected: MPS (Apple Silicon)")
    elif torch.cuda.is_available():
        device = torch.device("cuda")
        print("Device Selected: CUDA")
    else:
        device = torch.device("cpu")
        print("Device Selected: CPU")
    return device

device = get_device()

Device Selected: MPS (Apple Silicon)


## 1. Load the LoRA-Finetuned Model & Tokenizer

We load the model (which already incorporates LoRA modules) in float16 mode to respect our memory constraints.


In [2]:
MODEL_LORA_DIR = "../checkpoints/model_lora"
print("Loading the LoRA-finetuned model from:", MODEL_LORA_DIR)

try:
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_LORA_DIR,
        torch_dtype=torch.float16,     # Load in lower precision for memory efficiency
        low_cpu_mem_usage=True         # Optimize memory usage during loading
    )
    tokenizer = AutoTokenizer.from_pretrained(MODEL_LORA_DIR)
    model.to(device)
    print("LoRA-finetuned model and tokenizer loaded successfully.")
except Exception as e:
    print("Error loading LoRA model/tokenizer:", e)
    raise

Loading the LoRA-finetuned model from: ../checkpoints/model_lora


Some weights of the model checkpoint at ../checkpoints/model_quantized were not used when initializing GPT2LMHeadModel: ['lm_head.scale', 'lm_head.zero_point']
- This IS expected if you are initializing GPT2LMHeadModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing GPT2LMHeadModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
/Users/swayamsingal/miniconda3/envs/research_env/lib/python3.10/site-packages/bitsandbytes/cextension.py:34: UserWarning: The installed version of bitsandbytes was compiled without GPU support. 8-bit optimizers, 8-bit multiplication, and GPU quantization are unavailable.
  warn("The installed version of bitsandbytes was compiled without GPU support. "


'NoneType' object has no attribute 'cadam32bit_grad_fp32'
LoRA-finetuned model and tokenizer loaded successfully.


## 2. Apply Pruning to the Model

We now traverse through the model and prune all layers of type `torch.nn.Linear` using L1 unstructured pruning.
You can adjust the pruning amount; here, we use an amount of 30% (i.e. 30% of weights are zeroed out).


In [3]:
import torch.nn.utils.prune as prune

def apply_pruning_to_module(module, amount=0.3):
    """
    Apply L1 unstructured pruning to the 'weight' parameter of a module.
    :param module: torch.nn module (expected to be torch.nn.Linear)
    :param amount: Fraction of connections to prune (0.3 means 30%)
    """
    try:
        prune.l1_unstructured(module, name="weight", amount=amount)
    except Exception as e:
        print(f"Pruning skipped for module due to error: {e}")

print("Applying pruning to all Linear layers in the model...")
for name, module in model.named_modules():
    if isinstance(module, torch.nn.Linear):
        print(f"Pruning layer: {name}")
        apply_pruning_to_module(module, amount=0.6)
print("Pruning complete.")

Applying pruning to all Linear layers in the model...
Pruning layer: transformer.h.0.attn.c_attn.lora_A.default
Pruning layer: transformer.h.0.attn.c_attn.lora_B.default
Pruning layer: transformer.h.1.attn.c_attn.lora_A.default
Pruning layer: transformer.h.1.attn.c_attn.lora_B.default
Pruning layer: transformer.h.2.attn.c_attn.lora_A.default
Pruning layer: transformer.h.2.attn.c_attn.lora_B.default
Pruning layer: transformer.h.3.attn.c_attn.lora_A.default
Pruning layer: transformer.h.3.attn.c_attn.lora_B.default
Pruning layer: transformer.h.4.attn.c_attn.lora_A.default
Pruning layer: transformer.h.4.attn.c_attn.lora_B.default
Pruning layer: transformer.h.5.attn.c_attn.lora_A.default
Pruning layer: transformer.h.5.attn.c_attn.lora_B.default
Pruning layer: transformer.h.6.attn.c_attn.lora_A.default
Pruning layer: transformer.h.6.attn.c_attn.lora_B.default
Pruning layer: transformer.h.7.attn.c_attn.lora_A.default
Pruning layer: transformer.h.7.attn.c_attn.lora_B.default
Pruning layer: tra

## 3. Save the Pruned Model Checkpoint

The pruned model and tokenizer are saved to a new directory (`../checkpoints/model_pruned`) for further processing.


In [10]:
OUTPUT_CHECKPOINT_DIR = "../checkpoints/model_pruned"
os.makedirs(OUTPUT_CHECKPOINT_DIR, exist_ok=True)
print("Saving the pruned model checkpoint...")

try:
    # Get state dict from the quantized model
    state_dict = model.state_dict()
    # Filter state dict to retain only tensor objects
    filtered_state_dict = {k: v for k, v in state_dict.items() if isinstance(v, torch.Tensor)}
    
    # Define the path for the saved state dict
    state_dict_path = os.path.join(OUTPUT_CHECKPOINT_DIR, "pytorch_model.bin")
    torch.save(filtered_state_dict, state_dict_path)
    print(f"pruned model state dict saved successfully in '{state_dict_path}'.")

    # Save the model configuration and tokenizer
    model.config.to_json_file(os.path.join(OUTPUT_CHECKPOINT_DIR, "config.json"))
    model.save_pretrained(OUTPUT_CHECKPOINT_DIR)
    tokenizer.save_pretrained(OUTPUT_CHECKPOINT_DIR)
    print("Model configuration and tokenizer saved successfully.")
except Exception as e:
    print(f"Error saving quantized checkpoint: {e}")
    raise

print(f"Pruned model checkpoint saved successfully in {OUTPUT_CHECKPOINT_DIR}")

Saving the pruned model checkpoint...
pruned model state dict saved successfully in '../checkpoints/model_pruned/pytorch_model.bin'.
Model configuration and tokenizer saved successfully.
Pruned model checkpoint saved successfully in ../checkpoints/model_pruned


In [32]:
model


PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): GPT2LMHeadModel(
      (transformer): GPT2Model(
        (wte): Embedding(50257, 768)
        (wpe): Embedding(1024, 768)
        (drop): Dropout(p=0.1, inplace=False)
        (h): ModuleList(
          (0-11): 12 x GPT2Block(
            (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
            (attn): GPT2Attention(
              (c_attn): lora.Linear(
                (base_layer): Conv1D(nf=2304, nx=768)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=768, out_features=8, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=8, out_features=2304, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitud

In [33]:
merged_model = model.merge_and_unload()

In [34]:
merged_model.save_pretrained(
    "../checkpoints/model_test",
    save_embedding_layers="none",  # Skip embedding layers
    safe_serialization=True,       # Use safetensors (smaller + safer)
)

UnboundLocalError: local variable 'active_adapters' referenced before assignment

## Completion Message

The 4_pruning notebook has successfully pruned the LoRA-finetuned model and saved the new checkpoint.
You may now proceed to further steps in your pipeline.



In [5]:
print("4_pruning notebook execution complete. Checkpoint stored at:", OUTPUT_CHECKPOINT_DIR)

4_pruning notebook execution complete. Checkpoint stored at: ../checkpoints/model_pruned
